# 🚀 Startup Success Prediction at Seed Stage

**Research Question:**  
> Can we predict, at the time of a startup's seed round, whether it will be **acquired**, remain **operating**, or **close**?

**Dataset:** VC Investments (Crunchbase-based) — 54,000+ startups, 39 features  
**Target Variable:** `status` — multi-class: `acquired` / `operating` / `closed`  
**Scope:** Only startups that received seed funding. We use **only features available at seed time** (no future round data).

---

### ⚠️ Important Methodological Note on 'Operating'
The `operating` class represents **censored data** — these companies have not failed *yet*, but their ultimate fate is unknown. We treat this class as *"survived so far"*, not *"will ultimately succeed"*. This should be kept in mind when interpreting model results.

---

### Project Workflow
1. Setup & Data Loading
2. Data Preparation & Leakage Prevention
3. Exploratory Data Analysis (EDA)
4. Data Cleansing
5. Feature Engineering
6. Categorical Encoding
7. Feature Selection
8. Modeling (Multi-class)
9. Model Evaluation
10. Binary View: Success vs Closed

## 1. Setup & Data Loading

In [3]:
# Install any extra libraries if needed (uncomment in Colab)
# !pip install lightgbm shap --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score, f1_score
)
from sklearn.utils.class_weight import compute_class_weight
import xgboost as xgb
import lightgbm as lgb

# Plotting settings
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13

print('✅ All libraries imported successfully')

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
# ── Load Data ──────────────────────────────────────────────────────────────
# If running in Colab, upload the file or load from Drive:
# from google.colab import files
# uploaded = files.upload()

df_raw = pd.read_csv('investments_VC.csv', encoding='latin1')

print(f'Dataset shape: {df_raw.shape}')
print(f'Columns: {df_raw.columns.tolist()}')
df_raw.head(3)

In [ ]:
# Quick overview of missing values and types
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(1)
pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct}).query('missing_count > 0').sort_values('missing_%', ascending=False)

## 2. Data Preparation & Leakage Prevention

**Key rule:** We can only use features that would be known **at the time of the seed round**.  
Columns like `round_A` through `round_H`, `venture`, `post_ipo_*` represent **future events** — using them would leak the answer.

In [ ]:
# Work on a copy
df = df_raw.copy()

# ── 1. Strip whitespace from column names ──────────────────────────────────
df.columns = df.columns.str.strip()

# ── 2. Drop future-leakage columns ─────────────────────────────────────────
leakage_cols = [
    'round_A', 'round_B', 'round_C', 'round_D',
    'round_E', 'round_F', 'round_G', 'round_H',
    'venture', 'post_ipo_equity', 'post_ipo_debt',
    'private_equity', 'secondary_market',
    'equity_crowdfunding', 'product_crowdfunding',
    'funding_rounds',   # total rounds — includes future rounds
    'last_funding_at',  # unknown at seed time
    'funding_total_usd' # total includes future funding
]

# Also drop identifier/URL columns (not predictive)
id_cols = ['permalink', 'name', 'homepage_url']

df = df.drop(columns=leakage_cols + id_cols)
print(f'Remaining columns ({len(df.columns)}): {df.columns.tolist()}')

In [ ]:
# ── 3. Keep only seed-stage companies ──────────────────────────────────────
# This is our study population: startups that actually raised a seed round
df = df[df['seed'] > 0].copy()
print(f'Rows after keeping seed > 0: {len(df)}')

# ── 4. Drop rows with no status label ──────────────────────────────────────
df = df.dropna(subset=['status'])
print(f'Rows after dropping null status: {len(df)}')

# ── 5. Target variable distribution ────────────────────────────────────────
print('\nTarget distribution:')
print(df['status'].value_counts())
print(f"\nClass imbalance ratio: {df['status'].value_counts().max() / df['status'].value_counts().min():.1f}x")

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# ── Target class distribution ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = df['status'].value_counts()
colors = ['#2ecc71', '#3498db', '#e74c3c']

axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Startup Status Distribution (Seed Companies)')
axes[0].set_xlabel('Status')
axes[0].set_ylabel('Count')
for i, (label, val) in enumerate(counts.items()):
    axes[0].text(i, val + 50, f'{val:,}', ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=counts.index, colors=colors,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Status Proportions')

plt.tight_layout()
plt.show()

In [ ]:
# ── Seed amount distribution by status ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Log scale seed amount
df['log_seed'] = np.log1p(df['seed'])

for status, color in zip(['operating', 'acquired', 'closed'], colors):
    subset = df[df['status'] == status]['log_seed']
    axes[0].hist(subset, bins=40, alpha=0.5, label=status, color=color)

axes[0].set_title('Log(Seed Amount) by Status')
axes[0].set_xlabel('log(seed amount)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Boxplot
df.boxplot(column='log_seed', by='status', ax=axes[1])
axes[1].set_title('Seed Amount by Status (log scale)')
axes[1].set_xlabel('Status')
axes[1].set_ylabel('log(seed)')
plt.suptitle('')

plt.tight_layout()
plt.show()

print('Median seed amount by status:')
print(df.groupby('status')['seed'].median().apply(lambda x: f'${x:,.0f}'))

In [ ]:
# ── Success rate by Top Industries ──────────────────────────────────────────
top_markets = df['market'].value_counts().head(12).index
market_df = df[df['market'].isin(top_markets)]

market_status = market_df.groupby(['market', 'status']).size().unstack(fill_value=0)
market_status_pct = market_status.div(market_status.sum(axis=1), axis=0) * 100

market_status_pct[['acquired', 'operating', 'closed']].plot(
    kind='bar', stacked=True, figsize=(14, 6),
    color=['#3498db', '#2ecc71', '#e74c3c'],
    edgecolor='white'
)
plt.title('Startup Outcome by Industry (Top 12 Markets)')
plt.xlabel('Market')
plt.ylabel('Percentage (%)')
plt.xticks(rotation=45, ha='right')
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
# ── Success rate by founding year ───────────────────────────────────────────
year_df = df.dropna(subset=['founded_year'])
year_df = year_df[year_df['founded_year'].between(2000, 2015)]  # focus on meaningful years

year_status = year_df.groupby(['founded_year', 'status']).size().unstack(fill_value=0)
year_status_pct = year_status.div(year_status.sum(axis=1), axis=0) * 100

year_status_pct[['acquired', 'operating', 'closed']].plot(
    kind='area', figsize=(14, 5), alpha=0.75,
    color=['#3498db', '#2ecc71', '#e74c3c']
)
plt.title('Outcome Distribution by Founding Year')
plt.xlabel('Founded Year')
plt.ylabel('Percentage (%)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Top countries by startup count & closure rate ───────────────────────────
top_countries = df['country_code'].value_counts().head(10).index
country_df = df[df['country_code'].isin(top_countries)]

country_closed = country_df[country_df['status'] == 'closed'].groupby('country_code').size()
country_total = country_df.groupby('country_code').size()
country_closure_rate = (country_closed / country_total * 100).sort_values(ascending=False)

country_closure_rate.plot(kind='bar', color='#e74c3c', alpha=0.8, figsize=(12, 4), edgecolor='white')
plt.title('Closure Rate by Country (Top 10 Countries by Volume)')
plt.ylabel('Closure Rate (%)')
plt.xlabel('Country Code')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation heatmap (numeric features) ───────────────────────────────────
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[numeric_cols].corr()

plt.figure(figsize=(12, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, linewidths=0.5, annot_kws={'size': 9})
plt.title('Correlation Matrix — Numeric Features')
plt.tight_layout()
plt.show()

## 4. Data Cleansing

In [ ]:
# ── Parse dates ─────────────────────────────────────────────────────────────
df['founded_at'] = pd.to_datetime(df['founded_at'], errors='coerce')
df['first_funding_at'] = pd.to_datetime(df['first_funding_at'], errors='coerce')

# ── Handle rare categories ───────────────────────────────────────────────────
# Collapse low-frequency markets into 'Other'
market_counts = df['market'].value_counts()
rare_markets = market_counts[market_counts < 100].index
df['market'] = df['market'].apply(lambda x: 'Other' if x in rare_markets else x)
df['market'] = df['market'].fillna('Unknown')
print(f'Unique markets after grouping rare: {df["market"].nunique()}')

# Same for country
country_counts = df['country_code'].value_counts()
rare_countries = country_counts[country_counts < 50].index
df['country_code'] = df['country_code'].apply(lambda x: 'Other' if x in rare_countries else x)
df['country_code'] = df['country_code'].fillna('Unknown')

# ── Handle outliers in seed ──────────────────────────────────────────────────
# Cap seed at 99th percentile to limit extreme outlier influence
seed_cap = df['seed'].quantile(0.99)
df['seed_capped'] = df['seed'].clip(upper=seed_cap)
print(f'Seed capped at 99th percentile: ${seed_cap:,.0f}')

# ── Missing values summary ───────────────────────────────────────────────────
print('\nMissing values after cleaning:')
print(df.isnull().sum()[df.isnull().sum() > 0])

## 5. Feature Engineering

Creating new features that capture startup-level signals available at the seed stage.

In [ ]:
# ── Time-based features ──────────────────────────────────────────────────────
# Days from founding to first funding (speed to raise)
df['days_to_first_funding'] = (df['first_funding_at'] - df['founded_at']).dt.days
# Cap extreme values (some companies have negative or >3000 days)
df['days_to_first_funding'] = df['days_to_first_funding'].clip(lower=0, upper=2000)

# Founding era buckets
def founding_era(year):
    if pd.isna(year): return 'Unknown'
    elif year < 2000: return 'Pre-2000'
    elif year < 2008: return '2000-2007'
    elif year < 2012: return '2008-2011'
    else: return '2012+'

df['founding_era'] = df['founded_year'].apply(founding_era)

# ── Seed funding features ────────────────────────────────────────────────────
df['log_seed'] = np.log1p(df['seed_capped'])

# Seed amount buckets
df['seed_bucket'] = pd.cut(
    df['seed'],
    bins=[0, 50_000, 250_000, 1_000_000, 5_000_000, float('inf')],
    labels=['micro', 'small', 'medium', 'large', 'mega']
)

# ── Funding type flags ───────────────────────────────────────────────────────
df['had_angel'] = (df['angel'] > 0).astype(int)
df['had_grant'] = (df['grant'] > 0).astype(int)
df['had_convertible'] = (df['convertible_note'] > 0).astype(int)
df['had_debt'] = (df['debt_financing'] > 0).astype(int)

# ── Geography features ───────────────────────────────────────────────────────
df['is_usa'] = (df['country_code'] == 'USA').astype(int)
df['is_silicon_valley'] = df['region'].str.lower().str.contains('san francisco|silicon valley', na=False).astype(int)
df['is_ny'] = df['region'].str.lower().str.contains('new york', na=False).astype(int)

# ── Category features ────────────────────────────────────────────────────────
# Extract primary category from category_list pipe-separated string
df['primary_category'] = df['category_list'].str.split('|').str[1]  # first real entry after leading |
df['primary_category'] = df['primary_category'].fillna('Unknown')

# Count number of categories a startup spans
df['num_categories'] = df['category_list'].str.count('\|')

print('✅ Feature engineering complete')
print('New features:', ['days_to_first_funding', 'founding_era', 'log_seed', 'seed_bucket',
                        'had_angel', 'had_grant', 'had_convertible', 'had_debt',
                        'is_usa', 'is_silicon_valley', 'is_ny', 'primary_category', 'num_categories'])

In [ ]:
# ── Visualize engineered features ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Days to first funding by status
df_notna = df.dropna(subset=['days_to_first_funding'])
df_notna.boxplot(column='days_to_first_funding', by='status', ax=axes[0])
axes[0].set_title('Days to First Funding by Status')
axes[0].set_ylabel('Days')
plt.suptitle('')

# Angel / grant flags by status
flag_data = df.groupby('status')[['had_angel', 'had_grant']].mean() * 100
flag_data.plot(kind='bar', ax=axes[1], color=['#f39c12', '#27ae60'], edgecolor='white')
axes[1].set_title('% with Angel / Grant Funding by Status')
axes[1].set_ylabel('Percentage (%)')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

## 6. Categorical Encoding & Final Feature Matrix

In [ ]:
# ── Select final features for modeling ───────────────────────────────────────
feature_cols = [
    # Seed funding
    'log_seed',
    # Funding type flags
    'had_angel', 'had_grant', 'had_convertible', 'had_debt',
    # Geography
    'is_usa', 'is_silicon_valley', 'is_ny',
    # Time-based
    'days_to_first_funding',
    'founded_year',
    # Category
    'num_categories',
    # Categorical (will be encoded below)
    'market', 'country_code', 'founding_era'
]

df_model = df[feature_cols + ['status']].copy()

# ── Fill remaining missing values ─────────────────────────────────────────────
df_model['days_to_first_funding'] = df_model['days_to_first_funding'].fillna(df_model['days_to_first_funding'].median())
df_model['founded_year'] = df_model['founded_year'].fillna(df_model['founded_year'].median())
df_model['num_categories'] = df_model['num_categories'].fillna(0)

# ── Encode categoricals with Label Encoding ────────────────────────────────
# NOTE: For high-cardinality features like market, target encoding would be
# better — we use label encoding here as a robust baseline.
cat_cols = ['market', 'country_code', 'founding_era']
le = LabelEncoder()
for col in cat_cols:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

# ── Encode target ──────────────────────────────────────────────────────────
le_target = LabelEncoder()
df_model['target'] = le_target.fit_transform(df_model['status'])
print('Target encoding:', dict(zip(le_target.classes_, le_target.transform(le_target.classes_))))

print(f'\nFinal model dataset shape: {df_model.shape}')
df_model.head(3)

## 7. Train / Validation / Test Split

We use **stratified splitting** to preserve class proportions across splits.  
The **test set is held out and only used once at the very end.**

In [ ]:
X = df_model.drop(columns=['status', 'target'])
y = df_model['target']

# First split: 80% train+val, 20% test (held out)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Second split: 75% train, 25% val (of the 80%)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=42
)

print(f'Train:      {X_train.shape[0]:,} rows ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Validation: {X_val.shape[0]:,} rows ({X_val.shape[0]/len(X)*100:.0f}%)')
print(f'Test:       {X_test.shape[0]:,} rows ({X_test.shape[0]/len(X)*100:.0f}%)')
print(f'\nClass distribution in train:')
print(pd.Series(y_train).map(dict(enumerate(le_target.classes_))).value_counts())

# ── Compute class weights for imbalanced training ──────────────────────────
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))
print(f'\nClass weights: {class_weight_dict}')

## 8. Modeling

We train 4 models and evaluate on the **validation set** (not test!).

In [ ]:
# ── Helper: evaluate on validation set ────────────────────────────────────
def evaluate_model(name, model, X_val, y_val, classes):
    y_pred = model.predict(X_val)
    f1 = f1_score(y_val, y_pred, average='weighted')
    print(f'\n{'='*50}')
    print(f'Model: {name}')
    print(f'Weighted F1 Score (Validation): {f1:.4f}')
    print('\nClassification Report:')
    print(classification_report(y_val, y_pred, target_names=classes))
    return f1

In [ ]:
# ── Model 1: Logistic Regression (Baseline) ────────────────────────────────
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        multi_class='multinomial',
        max_iter=1000,
        class_weight='balanced',
        random_state=42
    ))
])

lr_pipeline.fit(X_train, y_train)
lr_f1 = evaluate_model('Logistic Regression', lr_pipeline, X_val, y_val, le_target.classes_)

In [ ]:
# ── Model 2: Random Forest ─────────────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
rf_f1 = evaluate_model('Random Forest', rf, X_val, y_val, le_target.classes_)

In [ ]:
# ── Model 3: XGBoost ───────────────────────────────────────────────────────
# Compute scale_pos_weight for each class (XGBoost uses sample weights)
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight('balanced', y_train)

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train, sample_weight=sample_weights)
xgb_f1 = evaluate_model('XGBoost', xgb_model, X_val, y_val, le_target.classes_)

In [ ]:
# ── Model 4: LightGBM ─────────────────────────────────────────────────────
lgb_model = lgb.LGBMClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    class_weight='balanced',
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgb_model.fit(X_train, y_train)
lgb_f1 = evaluate_model('LightGBM', lgb_model, X_val, y_val, le_target.classes_)

## 9. Model Comparison & Confusion Matrices

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost', 'LightGBM'],
    'Weighted F1 (Val)': [lr_f1, rf_f1, xgb_f1, lgb_f1]
}).sort_values('Weighted F1 (Val)', ascending=False)

print('\n📊 Model Comparison:')
print(results.to_string(index=False))

best_model_name = results.iloc[0]['Model']
print(f'\n🏆 Best model: {best_model_name}')

In [ ]:
# ── Confusion Matrices ──────────────────────────────────────────────────────
models = [
    ('Logistic Regression', lr_pipeline),
    ('Random Forest', rf),
    ('XGBoost', xgb_model),
    ('LightGBM', lgb_model)
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, (name, model) in enumerate(models):
    y_pred = model.predict(X_val)
    cm = confusion_matrix(y_val, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le_target.classes_)
    disp.plot(ax=axes[i], colorbar=False)
    axes[i].set_title(name)

plt.suptitle('Confusion Matrices — Validation Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Feature Importance

In [ ]:
# ── Feature Importance from Random Forest & LightGBM ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (name, model) in zip(axes, [('Random Forest', rf), ('LightGBM', lgb_model)]):
    importances = model.feature_importances_
    feat_imp = pd.Series(importances, index=X_train.columns).sort_values(ascending=True).tail(12)
    feat_imp.plot(kind='barh', ax=ax, color='#3498db', edgecolor='white')
    ax.set_title(f'Top Feature Importances — {name}')
    ax.set_xlabel('Importance')

plt.tight_layout()
plt.show()

In [ ]:
# ── Optional: SHAP Values (requires shap library) ──────────────────────────
# Uncomment to run — may be slow on large datasets

# import shap
# explainer = shap.TreeExplainer(lgb_model)
# shap_values = explainer.shap_values(X_val)
# 
# # Summary plot for 'closed' class (index depends on le_target.classes_)
# closed_idx = list(le_target.classes_).index('closed')
# shap.summary_plot(shap_values[closed_idx], X_val, plot_type='bar', 
#                   feature_names=X_val.columns.tolist())

## 11. Final Evaluation on Test Set

> ⚠️ **Run this only once, at the very end.** The test set must never be used for tuning.

In [ ]:
# Replace 'best_model' with whichever model won above (e.g., lgb_model, xgb_model)
best_model = lgb_model  # ← update this based on your validation results

y_test_pred = best_model.predict(X_test)
test_f1 = f1_score(y_test, y_test_pred, average='weighted')

print('🏁 FINAL TEST SET EVALUATION')
print(f'Best Model: {best_model_name}')
print(f'Weighted F1 Score (Test): {test_f1:.4f}')
print()
print(classification_report(y_test, y_test_pred, target_names=le_target.classes_))

# Final confusion matrix
cm = confusion_matrix(y_test, y_test_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le_target.classes_)
disp.plot(colorbar=True)
plt.title(f'Final Confusion Matrix — {best_model_name} (Test Set)')
plt.tight_layout()
plt.show()

## 12. Binary View: Success vs Closed

Collapsing the multi-class to a binary problem: **Success** (acquired + operating) vs **Closed**.  
This gives a cleaner business-readable interpretation.

In [ ]:
# Map multi-class target to binary
label_map = {'acquired': 1, 'operating': 1, 'closed': 0}
y_binary_train = pd.Series(y_train).map(lambda x: label_map[le_target.classes_[x]])
y_binary_val   = pd.Series(y_val).map(lambda x: label_map[le_target.classes_[x]])
y_binary_test  = pd.Series(y_test).map(lambda x: label_map[le_target.classes_[x]])

print('Binary distribution (train):')
print(y_binary_train.value_counts())

# Train a binary LightGBM
lgb_binary = lgb.LGBMClassifier(
    n_estimators=300, max_depth=8,
    learning_rate=0.05, class_weight='balanced',
    random_state=42, n_jobs=-1, verbose=-1
)
lgb_binary.fit(X_train, y_binary_train)

y_binary_pred = lgb_binary.predict(X_val)
y_binary_proba = lgb_binary.predict_proba(X_val)[:, 1]

print(f'\nBinary F1 (Validation): {f1_score(y_binary_val, y_binary_pred):.4f}')
print(f'ROC-AUC (Validation):   {roc_auc_score(y_binary_val, y_binary_proba):.4f}')
print()
print(classification_report(y_binary_val, y_binary_pred, target_names=['Closed', 'Success']))

In [ ]:
# ── ROC Curve ─────────────────────────────────────────────────────────────
from sklearn.metrics import RocCurveDisplay

RocCurveDisplay.from_predictions(
    y_binary_val, y_binary_proba,
    name='LightGBM Binary',
    color='#e74c3c'
)
plt.plot([0, 1], [0, 1], 'k--', label='Random baseline')
plt.title('ROC Curve — Success vs Closed')
plt.legend()
plt.tight_layout()
plt.show()

## 13. Summary & Conclusions

**Fill this section in after running all the cells above:**

### Key Findings
- *Which features were most predictive of startup outcome?*
- *Did seed amount size matter? Geography? Industry?*
- *How well did the model distinguish acquired vs closed companies?*

### Limitations
1. **Censored 'operating' class** — these startups' final outcomes are unknown.
2. **Survivorship bias** — companies that never raised seed funding are not in this dataset.
3. **Missing founding dates** — ~30% of rows had missing `founded_at`, affecting time-based features.
4. **Label imbalance** — the dataset is dominated by `operating` startups, making `acquired` and `closed` harder to predict precisely.

### Next Steps
- Try target encoding for high-cardinality `market` and `country_code`
- Hyperparameter tuning with Optuna or GridSearchCV
- SMOTE or other oversampling for minority classes
- Add SHAP analysis for interpretability
- Consider survival analysis as an alternative framing